# Imports and Setup


In [ ]:

import pandas as pd
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

version = "3"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [2]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

### Check Fiscal Year


In [3]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
current_fy_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [4]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in current_fy_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

2027
FY27

2026
FY26


In [5]:
where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")

orderby_arr = [
    "agency_number", # 2: 
    "unit_appropriation_number", # 4: 
    "responsibility_center_code", # 23: 
    "budget_code_number", # 6: 
    "object_class_number", # 8: 
    "object_code", # 10: 
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")

where_string:
fiscal_year IN (2027)

orderby_string:
agency_number, unit_appropriation_number, responsibility_center_code, budget_code_number, object_class_number, object_code, publication_date DESC



### Query Data if necessary


In [6]:
data = []
offset = 0

limit = 1000

try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}_v{version}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            # select=select_string,
            where=where_string,
            # group=groupby_string,
            order=orderby_string,
            # order=":id",
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
        
        # break
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv", index=False)
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")

Querying for fiscal year 2027_v2:

Fetching rows starting at offset: 0
Fetching rows starting at offset: 1000
Fetching rows starting at offset: 2000
Fetching rows starting at offset: 3000
Fetching rows starting at offset: 4000
Fetching rows starting at offset: 5000
Fetching rows starting at offset: 6000
Fetching rows starting at offset: 7000
Fetching rows starting at offset: 8000
Fetching rows starting at offset: 9000
Fetching rows starting at offset: 10000
Fetching rows starting at offset: 11000
Fetching rows starting at offset: 12000
Fetching rows starting at offset: 13000
Fetching rows starting at offset: 14000
Fetching rows starting at offset: 15000
Fetching rows starting at offset: 16000
Fetching rows starting at offset: 17000
Fetching rows starting at offset: 18000
Fetching rows starting at offset: 19000
Fetching rows starting at offset: 20000
Fetching rows starting at offset: 21000
Fetching rows starting at offset: 22000
Fetching rows starting at offset: 23000
Fetching rows star

/var/folders/kq/xpb5g0qn23g4s4hvlvhwdgn80000gn/T/ipykernel_90700/944054463.py:44: DtypeWarning: Columns (0: agency_number, 1: responsibility_center_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")


In [7]:
len(df)

65000

In [12]:
df[df[[
    'publication_date',
    'fiscal_year',
    'agency_number',
    'agency_name',
    'unit_appropriation_number',
    'unit_appropriation_name',
    'responsibility_center_code',
    'responsibility_center_name',
    'budget_code_number',
    'budget_code_name',
    'object_class_number',
    'object_class_name',
    'object_code',
    'object_code_name',
    # 'intra_city_purchase_code',
    # 'personal_service_other_than_personal_service_indicator',
    # 'financial_plan_savings_flag',
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount',
    'adopted_budget_position',
    'current_modified_budget_position',
    'financial_plan_position',
    'adopted_budget_number_of_contracts',
    'current_modified_budget_number_of_contracts',
    'financial_plan_number_of_contracts',
]].duplicated(keep=False)]

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,adopted_budget_position,current_modified_budget_position,financial_plan_position,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts,intra_city_purchase_code
20017,20260512,2027,69,DEPARTMENT OF SOCIAL SERVICES,101,ADMINISTRATION-OTPS,9927,Immigrant Affairs AOTPS,40,OTHER SERVICES AND CHARGES,...,0,500000,0,0,0,0,0,0,0,37.0
20018,20260512,2027,69,DEPARTMENT OF SOCIAL SERVICES,101,ADMINISTRATION-OTPS,9927,Immigrant Affairs AOTPS,40,OTHER SERVICES AND CHARGES,...,0,500000,0,0,0,0,0,0,0,38.0
20020,20260217,2027,69,DEPARTMENT OF SOCIAL SERVICES,101,ADMINISTRATION-OTPS,9927,Immigrant Affairs AOTPS,40,OTHER SERVICES AND CHARGES,...,0,200000,0,0,0,0,0,0,0,38.0
20021,20260217,2027,69,DEPARTMENT OF SOCIAL SERVICES,101,ADMINISTRATION-OTPS,9927,Immigrant Affairs AOTPS,40,OTHER SERVICES AND CHARGES,...,0,200000,0,0,0,0,0,0,0,39.0
61259,20260512,2027,856,DEPARTMENT OF CITYWIDE ADMIN SERVICE,790,ENERGY MANAGEMENT - OTPS,Z903,Agency Chief Decarbonization Officer,40,OTHER SERVICES AND CHARGES,...,0,180000,0,0,0,0,0,0,0,40.0
61262,20260512,2027,856,DEPARTMENT OF CITYWIDE ADMIN SERVICE,790,ENERGY MANAGEMENT - OTPS,Z903,Agency Chief Decarbonization Officer,40,OTHER SERVICES AND CHARGES,...,0,180000,0,0,0,0,0,0,0,819.0
61267,20260217,2027,856,DEPARTMENT OF CITYWIDE ADMIN SERVICE,790,ENERGY MANAGEMENT - OTPS,Z903,Agency Chief Decarbonization Officer,40,OTHER SERVICES AND CHARGES,...,0,180000,0,0,0,0,0,0,0,40.0
61268,20260217,2027,856,DEPARTMENT OF CITYWIDE ADMIN SERVICE,790,ENERGY MANAGEMENT - OTPS,Z903,Agency Chief Decarbonization Officer,40,OTHER SERVICES AND CHARGES,...,0,180000,0,0,0,0,0,0,0,819.0


In [14]:
for i, col in enumerate(df.columns):
    print(f"{i}: '{col}',")

0: 'publication_date',
1: 'fiscal_year',
2: 'agency_number',
3: 'agency_name',
4: 'unit_appropriation_number',
5: 'unit_appropriation_name',
6: 'budget_code_number',
7: 'budget_code_name',
8: 'object_class_number',
9: 'object_class_name',
10: 'object_code',
11: 'object_code_name',
12: 'responsibility_center_code',
13: 'responsibility_center_name',
14: 'personal_service_other_than_personal_service_indicator',
15: 'financial_plan_savings_flag',
16: 'adopted_budget_amount',
17: 'current_modified_budget_amount',
18: 'financial_plan_amount',
19: 'adopted_budget_position',
20: 'current_modified_budget_position',
21: 'financial_plan_position',
22: 'adopted_budget_number_of_contracts',
23: 'current_modified_budget_number_of_contracts',
24: 'financial_plan_number_of_contracts',
25: 'intra_city_purchase_code',


In [15]:
df = df.sort_values(
    [
        "agency_number",
        "unit_appropriation_number",
        "responsibility_center_code",
        "budget_code_number",
        "object_class_number",
        "object_code",
        "publication_date"
        ],
    ascending=[True, True, True, True, True, True, False]
    ).reset_index(drop=True)

df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,adopted_budget_position,current_modified_budget_position,financial_plan_position,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts,intra_city_purchase_code
0,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0229,Counsel to the Mayor,1,FULL TIME SALARIED,...,1789231,1789231,1789231,8,8,8,0,0,0,NaN
1,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0229,Counsel to the Mayor,1,FULL TIME SALARIED,...,1789231,1789231,1789231,8,8,8,0,0,0,NaN
2,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0230,Mayor's Judiciary Committee,1,FULL TIME SALARIED,...,88404,237404,88404,1,1,1,0,0,0,NaN
3,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0230,Mayor's Judiciary Committee,1,FULL TIME SALARIED,...,88404,168404,88404,1,1,1,0,0,0,NaN
4,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0245,Comm to Combat Domestic Violence,1,FULL TIME SALARIED,...,1870336,1870336,1870336,12,12,12,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64995,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,P010,Fleet Reduction,40,OTHER SERVICES AND CHARGES,...,0,0,-1060000000,0,0,0,0,0,0,NaN
64996,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,...,0,0,-400000000,0,0,0,0,0,0,NaN
64997,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,...,0,0,37689973,0,0,0,0,0,0,NaN
64998,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,Tier IV RECs,40,OTHER SERVICES AND CHARGES,...,0,0,75000000,0,0,0,0,0,0,NaN


In [17]:
# Reorder columns
df = df[
    [
    "publication_date", # 0: 
    "fiscal_year", # 1: 
    "agency_number", # 2: 
    "agency_name", # 3: 
    "unit_appropriation_number", # 4: 
    "unit_appropriation_name", # 5: 
    "responsibility_center_code", # 23: 
    "responsibility_center_name", # 24: 
    "budget_code_number", # 6: 
    "budget_code_name", # 7: 
    "object_class_number", # 8: 
    "object_class_name", # 9: 
    "object_code", # 10: 
    "object_code_name", # 11: 
    "intra_city_purchase_code", # 25: 
    "personal_service_other_than_personal_service_indicator", # 12: 
    "financial_plan_savings_flag", # 13: 
    "adopted_budget_amount", # 14: 
    "current_modified_budget_amount", # 15: 
    "financial_plan_amount", # 16: 
    "adopted_budget_position", # 17: 
    "current_modified_budget_position", # 18: 
    "financial_plan_position", # 19: 
    "adopted_budget_number_of_contracts", # 20: 
    "current_modified_budget_number_of_contracts", # 21: 
    "financial_plan_number_of_contracts", # 22: 
    ]
]
df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,...,financial_plan_savings_flag,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,adopted_budget_position,current_modified_budget_position,financial_plan_position,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts
0,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,...,N,1789231,1789231,1789231,8,8,8,0,0,0
1,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,...,N,1789231,1789231,1789231,8,8,8,0,0,0
2,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,...,N,88404,237404,88404,1,1,1,0,0,0
3,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,...,N,88404,168404,88404,1,1,1,0,0,0
4,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,Comm to Combat Domestic Violence,...,N,1870336,1870336,1870336,12,12,12,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64995,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,NaN,NaN,P010,Fleet Reduction,...,Y,0,0,-1060000000,0,0,0,0,0,0
64996,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,NaN,NaN,P002,FINANCIAL PLAN SAVINGS,...,Y,0,0,-400000000,0,0,0,0,0,0
64997,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,NaN,NaN,P002,FINANCIAL PLAN SAVINGS,...,Y,0,0,37689973,0,0,0,0,0,0
64998,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,NaN,NaN,P003,Tier IV RECs,...,Y,0,0,75000000,0,0,0,0,0,0


## Track releases


In [18]:
pub_dates = df["publication_date"].sort_values(ascending=True).unique().tolist()

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

2 pub_dates in FY 2027: [20260217, 20260512]


In [19]:
df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Raw",index=False)

## Set up base DataFrame and map function


In [ ]:
numeric_cols = [
    "adopted_budget_amount",
    "current_modified_budget_amount",
    "financial_plan_amount",
    
    "adopted_budget_position",
    "current_modified_budget_position",
    "financial_plan_position",
    
    "adopted_budget_number_of_contracts",
    "current_modified_budget_number_of_contracts",
    "financial_plan_number_of_contracts"
]

categorical_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    "financial_plan_savings_flag",
    "intra_city_purchase_code",
    "personal_service_other_than_personal_service_indicator",
]

base_df = df[categorical_cols].drop_duplicates().reset_index(drop=True)
base_df.head()

In [ ]:
# AGENCY_MAP = df[["agency_number","agency_name"]].drop_duplicates().set_index('agency_number')['agency_name'].to_dict()

# UOA_MAP = df.groupby('agency_number')[['unit_appropriation_number', 'unit_appropriation_name']].apply(
#     lambda x: dict(zip(x['unit_appropriation_number'], x['unit_appropriation_name']))
# ).to_dict()

# BUDGETCODE_MAP = df.groupby(['agency_number', 'unit_appropriation_number'])[["budget_code_number","budget_code_name"]].apply(lambda x: dict(zip(x['budget_code_number'], x['budget_code_name']))).to_dict()

# BUDGETCODE_MAP

## Loop through each release


In [ ]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

## Collapse Releases


In [ ]:
collapsed_df = base_df.copy()

new_numeric_cols = []


start_of_numerical_cols = len(categorical_cols)

# collapsed_df.head()

for i, pub_date in enumerate(pub_dates):
    ith_release_df = df[df["publication_date"] == pub_date][categorical_cols + numeric_cols].copy()
    
    print(i)
    
    # print(ith_release_df.columns[len(categorical_cols):])
    
    if i <= 2:
        for col in ["financial_plan_amount", "financial_plan_position", "financial_plan_number_of_contracts"]:
            # print(col)
            ith_release_df = ith_release_df.rename(
                columns={
                    f"{col}":f"{col}_{RELEASE_NAMES[i]}",
                    }
            )
            # print()
            # print(ith_release_df.columns)
    
        if i < num_pubs_this_year - 1:
            ith_release_df = ith_release_df.drop(columns=[
                f"adopted_budget_amount",
                f"current_modified_budget_amount",
                f"adopted_budget_position",
                f"current_modified_budget_position",
                f"adopted_budget_number_of_contracts",
                f"current_modified_budget_number_of_contracts",
                ])
            
        new_numeric_cols.extend(ith_release_df.columns[start_of_numerical_cols:])
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    # print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=categorical_cols, how='left')
    
    # break

collapsed_df.columns

In [ ]:
new_numeric_cols

In [ ]:
print(collapsed_df.columns)

collapsed_df

In [ ]:
collapsed_df[collapsed_df.duplicated(keep=False)]

## Budget Code Level


In [ ]:
Budget_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    "financial_plan_savings_flag",
]

main_cols = [
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount_exec'
]

budget_code_collapsed_df = collapsed_df.groupby(Budget_Code_cols,dropna=False).sum().reset_index()

budget_code_collapsed_df = budget_code_collapsed_df[Budget_Code_cols + new_numeric_cols]

budget_code_collapsed_df[Budget_Code_cols + main_cols]


## Responsibility Center Level


In [ ]:
RC_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    "financial_plan_savings_flag",
]

RC_collapsed_df = collapsed_df.groupby(RC_cols,dropna=False).sum().reset_index()

RC_collapsed_df = RC_collapsed_df[RC_cols + new_numeric_cols]

# RC_collapsed_df[RC_cols + main_cols]
RC_collapsed_df[RC_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][RC_cols + main_cols]

## Unit of Appropriation Level


In [ ]:
UoA_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

UoA_collapsed_df = collapsed_df.groupby(UoA_cols,dropna=False).sum().reset_index()

UoA_collapsed_df = UoA_collapsed_df[UoA_cols + new_numeric_cols]
UoA_collapsed_df[UoA_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][UoA_cols + main_cols]

## Agency Level


In [ ]:
Agency_cols = [
    "agency_number",
    "agency_name",
    # "unit_appropriation_number",
    # "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

Agency_collapsed_df = collapsed_df.groupby(Agency_cols,dropna=False).sum().reset_index()

Agency_collapsed_df = Agency_collapsed_df[Agency_cols + new_numeric_cols]

Agency_collapsed_df[Agency_cols + main_cols]